In [1]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import os
import numpy as np
import seaborn as sns

In [2]:
username = os.path.expanduser('~')
username

'/home/ehdgus7279'

In [3]:
# 소수점 아래 10자리까지 출력
pd.options.display.float_format = '{:.10f}'.format

In [6]:
ns = 1e-9
xDim=4
yDim=4

In [7]:
base_dir=f'/home/ehdgus7279/mgvm/results/memory-traces'
date=f'/20260429-210946-mesh'
trace_dir=f'{base_dir}{date}'

In [9]:
benchmark = 'gups'
stats_file = f"{trace_dir}/{benchmark}_mesh_{xDim}x{yDim}_memory_trace.csv"

df = pd.read_csv(stats_file, skipinitialspace=True)
df

,time,addr,pc,owner,rw,rdma,vector,type,chiplet
0,0.0490569970,3357581312,1207959808,0,0,0,0,2,0
1,0.0490569970,3357581312,1207959808,0,0,0,0,0,0
2,0.0490569990,3357581312,1207959808,0,0,0,0,0,0
3,0.0490570010,3357581312,1207959808,0,0,0,0,0,0
4,0.0490570030,3357581312,1207959808,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
19195014,0.0498482690,2147487872,1207959948,0,1,1,1,4,14
19195015,0.0498482690,2148274176,1207960020,12,0,1,1,1,7
19195016,0.0498482690,2147487744,1207959948,0,1,1,1,4,3
19195017,0.0498482690,2147487744,1207959948,0,1,1,1,4,4


In [11]:
# 1. 주소별 고유 칩렛 개수 및 접근 횟수 집계
shared_chiplets = df.groupby(['pc', 'addr']).agg({
    'addr': 'count',    # 해당 주소로의 총 접근 횟수
    'chiplet': 'nunique' # 공유된 칩렛 수
}).rename(columns={
    'addr': 'access_count', 
    'chiplet': 'unique_chiplets'
}).reset_index()

In [12]:
shared_chiplets

,pc,addr,access_count,unique_chiplets
0,1207959808,3357581312,80,16
1,1207959852,3357585408,80,16
2,1207959928,3224440320,1,1
3,1207959928,3224440384,1,1
4,1207959928,3224440448,1,1
...,...,...,...,...
167483,1207960048,3147755520,1229,16
167484,1207960048,3147829248,339,16
167485,1207960048,3147843584,11,10
167486,1207960048,3151007744,1207,16


In [21]:
def _generate_dynamic_bins(max_chiplets):
    """
    최대 칩렛 수를 기반으로 동적으로 bin 경계와 라벨 생성.
    """
    if max_chiplets <= 1:
        return [1, np.inf], ['1']
    
    bins = [1]
    labels = ['1']
    
    power = 2
    while power <= max_chiplets:
        bins.append(power)
        next_power = power * 2
        
        # Calculate max value in this bin
        bin_max = next_power - 1
        
        if next_power > max_chiplets:
            labels.append(f'{power}+')
            bins.append(np.inf)
            break
        else:
            if power == bin_max:
                labels.append(f'{power}')
            else:
                labels.append(f'{power}-{bin_max}')
        
        power = next_power
        
    return bins, labels


def process_benchmark(benchmark, args):
    trace_file = f"{trace_dir}/{benchmark}_mesh_{xDim}x{yDim}_memory_trace.csv"

    if not os.path.exists(trace_file):
        return None
    
    print(f"[{benchmark}] Analyzing chiplet sharing by PC...")
    
    try:
        # NOTE: pc 정보를 포함합니다.
        df = pd.read_csv(trace_file, 
                        skipinitialspace=True, 
                        usecols=['pc', 'addr', 'chiplet', 'rdma', 'owner'])
        
        # RDMA 요청 필터링 여부 (기존 방식과 동일하게 유지)
        df = df[df['rdma']==True]

        # if args.group_size != 0:
        #     group_size = args.group_size
        #     df['addr'] = df['addr'] // (64 * group_size)
        
        if df.empty:
            return None

        # 1. PC 및 addr 조합별로 칩렛 개수 접근 횟수 집계
        pc_addr_df = df.groupby(['pc', 'addr']).agg(
            access_count=('chiplet', 'count'),     # 해당 주소로의 총 접근 횟수 (특정 PC 기준)
            unique_chiplets=('chiplet', 'nunique') # 공유된 칩렛 수
        ).reset_index()

        if pc_addr_df.empty:
            return None

        max_chiplets = pc_addr_df['unique_chiplets'].max()
        bins, labels = _generate_dynamic_bins(max_chiplets)

        # 각 (pc, addr) 구간 분류
        pc_addr_df['range'] = pd.cut(pc_addr_df['unique_chiplets'], bins=bins, labels=labels, right=False)

        # 2. PC 단위로 요약 메트릭 생성
        pc_stats = pc_addr_df.groupby('pc').agg(
            total_addrs=('addr', 'count'),
            total_accesses=('access_count', 'sum'),
            avg_sharing=('unique_chiplets', 'mean'),
            max_sharing=('unique_chiplets', 'max')
        )

        # 3. PC 별 분포 계산 (Address Count 기준 - %)
        addr_dist = pd.crosstab(pc_addr_df['pc'], pc_addr_df['range'], dropna=False)
        addr_dist_pct = addr_dist.div(addr_dist.sum(axis=1), axis=0) * 100
        addr_dist_pct.columns = [f'Addr_%_{label}' for label in labels]

        # 4. PC 별 분포 계산 (Access Count 기준 - %)
        acc_dist = pc_addr_df.pivot_table(index='pc', columns='range', values='access_count', aggfunc='sum', observed=False).fillna(0)
        acc_dist_pct = acc_dist.div(acc_dist.sum(axis=1), axis=0) * 100
        acc_dist_pct.columns = [f'Access_%_{label}' for label in labels]

        # 5. 병합
        res_df = pc_stats.join(addr_dist_pct).join(acc_dist_pct).reset_index()
        res_df.insert(0, 'benchmark', benchmark)

        # 접근이 많은 PC 순으로 정렬
        res_df = res_df.sort_values(by='total_accesses', ascending=False)
        
        # 반올림
        numeric_cols = res_df.select_dtypes(include=[np.number]).columns
        res_df[numeric_cols] = res_df[numeric_cols].round(2)

        return res_df

    except Exception as e:
        print(f"[{benchmark}] Error: {e}")
        return None

In [22]:
result_df = process_benchmark(benchmark, args=None)

result_df

[gups] Analyzing chiplet sharing by PC...


,benchmark,pc,total_addrs,total_accesses,avg_sharing,max_sharing,Addr_%_1,Addr_%_2-3,Addr_%_4-7,Addr_%_8+,Access_%_1,Access_%_2-3,Access_%_4-7,Access_%_8+
2,gups,1207959948,463,16349225,14.5900000000,15,0.0000000000,0.0000000000,1.7300000000,98.2700000000,0.0000000000,0.0000000000,0.0500000000,99.9500000000
5,gups,1207960024,462,686242,10.3100000000,15,1.7300000000,7.5800000000,26.1900000000,64.5000000000,0.0000000000,0.0100000000,0.1900000000,99.8000000000
11,gups,1207960048,456,232698,10.2900000000,15,1.3200000000,6.1400000000,29.1700000000,63.3800000000,0.0000000000,0.0400000000,0.4000000000,99.5600000000
4,gups,1207960020,215,231973,13.9700000000,15,2.3300000000,1.8600000000,2.3300000000,93.4900000000,0.0000000000,0.0000000000,0.0400000000,99.9500000000
6,gups,1207960028,457,224414,13.7400000000,15,0.2200000000,0.6600000000,2.8400000000,96.2800000000,0.0000000000,0.0000000000,0.1300000000,99.8700000000
8,gups,1207960036,359,22762,9.2000000000,15,23.6800000000,13.9300000000,3.9000000000,58.5000000000,0.4000000000,0.5100000000,0.4300000000,98.6600000000
7,gups,1207960032,362,22250,9.0700000000,15,21.2700000000,18.2300000000,2.7600000000,57.7300000000,0.3600000000,0.7000000000,0.4400000000,98.5000000000
9,gups,1207960040,357,19569,9.0700000000,15,24.6500000000,15.9700000000,0.8400000000,58.5400000000,0.4600000000,0.6600000000,0.2200000000,98.6600000000
10,gups,1207960044,325,14503,9.3700000000,15,27.3800000000,7.3800000000,4.0000000000,61.2300000000,0.6300000000,0.3700000000,1.1700000000,97.8300000000
3,gups,1207960016,159,483,2.7200000000,11,28.3000000000,46.5400000000,23.2700000000,1.8900000000,9.7300000000,39.5400000000,39.7500000000,10.9700000000
